In [8]:
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt

# Step 1: Ingestion and Naive Model
url = 'https://raw.githubusercontent.com/xingtianma/econ3916/refs/heads/main/data/Zillow_California_2026_Hedonic.csv'
df = pd.read_csv(url)

naive_model = smf.ols("Sale_Price ~ Property_Age", data=df).fit()
print(naive_model.summary())
print("\nNaive Age Coefficient:", naive_model.params['Property_Age'])

# Step 2: The Multivariate Model
multi_model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df).fit()
print(multi_model.summary())
print("\nMultivariate Age Coefficient:", multi_model.params['Property_Age'])

# Step 3: FWL Theorem Manual Proof 
# 3a: Partial out distance from Price
res_y_model = smf.ols('Sale_Price ~ Distance_to_Tech_Hub', data=df).fit()
df['Price_Residuals'] = res_y_model.resid

# 3b: Partial out distance from Age
res_x_model = smf.ols('Property_Age ~ Distance_to_Tech_Hub', data=df).fit()
df['Age_Residuals'] = res_x_model.resid

# 3c: Regress Residuals on Residuals (-1 removes the intercept for exact mathematical matching)
fwl_model = smf.ols('Price_Residuals ~ Age_Residuals', data=df).fit()
print("\nFWL Isolated Age Coefficient:", fwl_model.params['Age_Residuals'])

                            OLS Regression Results                            
Dep. Variable:             Sale_Price   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.757
Method:                 Least Squares   F-statistic:                     3105.
Date:                Thu, 26 Mar 2026   Prob (F-statistic):          1.26e-308
Time:                        21:03:13   Log-Likelihood:                -12818.
No. Observations:                1000   AIC:                         2.564e+04
Df Residuals:                     998   BIC:                         2.565e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
Intercept     3.013e+05   7218.570     41.742   

In [9]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import plotly.graph_objects as go

# ==========================================
# 1. MOCK DATA & MODEL SETUP 
# ==========================================
np.random.seed(42)
n_samples = 300
age = np.random.uniform(1, 50, n_samples)
distance = np.random.uniform(0.5, 30, n_samples)

# Simulating a negative relationship for both variables, plus some noise
true_price = 800000 - (5000 * age) - (10000 * distance) + np.random.normal(0, 50000, n_samples)

df = pd.DataFrame({
    'Sale_Price': true_price, 
    'Property_Age': age, 
    'Distance_to_Tech_Hub': distance
})

# Fit the OLS model
model = smf.ols('Sale_Price ~ Property_Age + Distance_to_Tech_Hub', data=df)
results = model.fit()

# ==========================================
# 2. MECHANISM CHECK: EXTRACTING COEFFICIENTS
# ==========================================
# We pull the specific betas from the results.params dictionary. 
# These represent the intercept and the slopes of the plane along the X and Y axes.
intercept = results.params['Intercept']
beta_age = results.params['Property_Age']
beta_distance = results.params['Distance_to_Tech_Hub']

# ==========================================
# 3. MECHANISM CHECK: GENERATING THE MESHGRID
# ==========================================
# To draw a plane, Plotly needs a 2D grid of X and Y coordinates, and a corresponding 
# 2D grid of Z heights. 

# First, we create 1D arrays representing the min to max range of our two independent variables.
x_range = np.linspace(df['Property_Age'].min(), df['Property_Age'].max(), 50)
y_range = np.linspace(df['Distance_to_Tech_Hub'].min(), df['Distance_to_Tech_Hub'].max(), 50)

# Next, np.meshgrid takes those two 1D arrays and broadcasts them against each other 
# to create two 2D matrices (x_grid and y_grid) representing every possible combination 
# of Age and Distance in that range.
x_grid, y_grid = np.meshgrid(x_range, y_range)

# Finally, we calculate the Z_grid (Predicted Sale Price) by plugging the 2D matrices 
# into our extracted OLS regression equation.
z_grid = intercept + (beta_age * x_grid) + (beta_distance * y_grid)

# ==========================================
# 4. PLOTLY GRAPH OBJECTS VISUALIZATION
# ==========================================
fig = go.Figure()

# Add the actual data points as a 3D scatter plot
fig.add_trace(go.Scatter3d(
    x=df['Property_Age'],
    y=df['Distance_to_Tech_Hub'],
    z=df['Sale_Price'],
    mode='markers',
    marker=dict(size=4, color='blue', opacity=0.6),
    name='Actual Data'
))

# Add the regression hyperplane using the meshgrid we generated
fig.add_trace(go.Surface(
    x=x_grid,
    y=y_grid,
    z=z_grid,
    colorscale='Viridis',
    opacity=0.7, # Keep it semi-transparent so we can see points above and below the plane
    name='OLS Hyperplane',
    showscale=False
))

# Update layout for frontend aesthetics
fig.update_layout(
    title='Multivariate OLS: 3D Regression Hyperplane',
    scene=dict(
        xaxis_title='Property Age (Years)',
        yaxis_title='Distance to Tech Hub (Miles)',
        zaxis_title='Sale Price ($)'
    ),
    width=900,
    height=700,
    margin=dict(l=0, r=0, b=0, t=50)
)

fig.show()